# 4D-Humans (HMR 2.0) Body Measurement Test

Extracts **30+ body measurements** (circumferences, widths, depths, lengths) from a single photo using:
- **4D-Humans / HMR 2.0** — state-of-the-art pose + shape estimation (ICCV 2023)
- **SMPL body model** — 6890-vertex 3D mesh from which any measurement can be sliced
- **trimesh** — mesh cross-section slicing for precise circumferences and widths

This is a standalone test. Once validated, the output format matches what `colab_server.py` expects.

---
### Before running
1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells top-to-bottom
3. In **Cell 7**, upload a full-body front-facing photo and enter the person's height

In [ ]:
# ── Cell 1: Verify GPU ──
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout.split('\n')[0])
    print('✅ GPU found')
else:
    print('❌ No GPU — Runtime > Change runtime type > GPU (T4)')

In [ ]:
# ── Cell 2: Install dependencies ──
# Takes ~3 minutes on first run
import sys
!{sys.executable} -m pip install -q git+https://github.com/shubham-goel/4D-Humans.git
!{sys.executable} -m pip install -q smplx trimesh einops yacs loguru pytorch-lightning hydra-core timm
!{sys.executable} -m pip install -q opencv-python-headless matplotlib pillow
print('✅ Dependencies installed')

In [ ]:
# ── Cell 3: Imports + download model weights ──
# download_models() fetches the HMR2 checkpoint + SMPL neutral model from HuggingFace (~600MB)
import os, json, warnings
import numpy as np
import torch
import cv2
import trimesh
import smplx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from torch.utils.data import DataLoader
warnings.filterwarnings('ignore')

from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import download_models, load_hmr2, DEFAULT_CHECKPOINT
from hmr2.utils import recursive_to
from hmr2.datasets.vitdet_dataset import ViTDetDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Downloading HMR2 checkpoint + SMPL model...')
download_models(CACHE_DIR_4DHUMANS)
print('✅ Model weights ready')

In [ ]:
# ── Cell 4: Load HMR2 model ──
print('Loading HMR2 model...')
model, model_cfg = load_hmr2(DEFAULT_CHECKPOINT)
model = model.to(device)
model.eval()
print('✅ HMR2 model loaded')

# Load separate SMPL neutral model for neutral-pose measurement reconstruction
SMPL_DIR = Path(CACHE_DIR_4DHUMANS) / 'data' / 'smpl'
print(f'SMPL model path: {SMPL_DIR}')
if not SMPL_DIR.exists():
    print('⚠️  SMPL dir not found — trying alternate path')
    # Try finding it in the cache
    for root, dirs, files in os.walk(CACHE_DIR_4DHUMANS):
        for f in files:
            if 'SMPL_NEUTRAL' in f or 'smpl_neutral' in f.lower():
                SMPL_DIR = Path(root)
                print(f'  Found SMPL at: {SMPL_DIR}')
                break

smpl_neutral = smplx.create(
    str(SMPL_DIR),
    model_type='smpl',
    gender='neutral',
    num_betas=10,
    batch_size=1
).to(device)
print('✅ SMPL neutral model loaded')

In [ ]:
# ── Cell 5: Measurement extraction functions ──

# SMPL 24 joint indices
J = {
    'pelvis': 0, 'l_hip': 1, 'r_hip': 2, 'spine1': 3,
    'l_knee': 4, 'r_knee': 5, 'spine2': 6,
    'l_ankle': 7, 'r_ankle': 8, 'spine3': 9,
    'l_foot': 10, 'r_foot': 11, 'neck': 12,
    'l_collar': 13, 'r_collar': 14, 'head': 15,
    'l_shoulder': 16, 'r_shoulder': 17,
    'l_elbow': 18, 'r_elbow': 19,
    'l_wrist': 20, 'r_wrist': 21,
    'l_hand': 22, 'r_hand': 23,
}


def _slice(mesh, origin, normal):
    """Returns line segments from mesh cross-section, or None."""
    try:
        lines = trimesh.intersections.mesh_plane(mesh, normal, origin)
        return lines if lines is not None and len(lines) > 0 else None
    except Exception:
        return None


def circumference(lines):
    if lines is None:
        return None
    return round(sum(np.linalg.norm(s[1] - s[0]) for s in lines) * 100, 1)


def width_depth(lines):
    """Returns (width_cm, depth_cm) = X-extent and Z-extent of cross-section."""
    if lines is None:
        return None, None
    pts = lines.reshape(-1, 3)
    w = round((pts[:, 0].max() - pts[:, 0].min()) * 100, 1)
    d = round((pts[:, 2].max() - pts[:, 2].min()) * 100, 1)
    return w, d


def extract_all_measurements(vertices, faces, joints, height_cm):
    """
    Extract all body measurements from SMPL mesh.

    Parameters
    ----------
    vertices : (6890, 3) float — SMPL vertex positions in meters
    faces    : (13776, 3) int  — triangle face indices
    joints   : (24+, 3) float — SMPL joint positions in meters
    height_cm: float           — actual person height for scaling

    Returns
    -------
    dict with keys: circumferences_cm, widths_cm, depths_cm, lengths_cm
    """
    jt = joints[:24].copy()

    # Scale mesh + joints so model height == actual height
    foot_y   = min(jt[J['l_foot']][1], jt[J['r_foot']][1])
    head_y   = jt[J['head']][1]
    model_h  = head_y - foot_y
    scale    = (height_cm / 100.0) / model_h if model_h > 0 else 1.0
    v = vertices * scale
    j = jt * scale

    mesh = trimesh.Trimesh(vertices=v, faces=faces, process=False)
    UP   = np.array([0.0, 1.0, 0.0])

    circs  = {}
    widths = {}
    depths = {}
    lens   = {}

    def measure_y(name, y_val):
        """Slice horizontally at y_val, record circumference + width + depth."""
        lines = _slice(mesh, np.array([0.0, y_val, 0.0]), UP)
        c = circumference(lines)
        w, d = width_depth(lines)
        if c: circs[name]  = c
        if w: widths[name] = w
        if d: depths[name] = d

    # ── Torso circumferences (horizontal slices) ──

    # Head: 3cm below head joint
    measure_y('head',  j[J['head']][1]  - 0.03)

    # Neck: 30% of the way from neck joint toward head joint
    neck_y = j[J['neck']][1] + 0.3 * (j[J['head']][1] - j[J['neck']][1])
    measure_y('neck',  neck_y)

    # Chest/Bust: 3% below shoulder joint midpoint (at armpit level)
    chest_y = (j[J['l_shoulder']][1] + j[J['r_shoulder']][1]) / 2 * 0.97
    measure_y('chest', chest_y)
    circs['bust'] = circs.get('chest')   # same measurement, different label

    # Waist: midpoint between spine2 and spine3
    measure_y('waist', (j[J['spine2']][1] + j[J['spine3']][1]) / 2)

    # Belly / navel: midpoint between spine2 and pelvis
    measure_y('belly', (j[J['spine2']][1] + j[J['pelvis']][1]) / 2)

    # Hips: 5cm below pelvis joint (widest hip region)
    measure_y('hips',  j[J['pelvis']][1] - 0.05)

    # ── Leg circumferences ──
    for side, hip_i, knee_i, ankle_i in [
        ('l', J['l_hip'], J['l_knee'], J['l_ankle']),
        ('r', J['r_hip'], J['r_knee'], J['r_ankle']),
    ]:
        # Upper thigh: 25% from hip toward knee
        measure_y(f'thigh_upper_{side}',
                  j[hip_i][1] + 0.25 * (j[knee_i][1] - j[hip_i][1]))

        # Mid thigh: 50% between hip and knee
        measure_y(f'thigh_{side}',
                  (j[hip_i][1] + j[knee_i][1]) / 2)

        # Knee upper: 2cm above knee joint
        measure_y(f'knee_upper_{side}',  j[knee_i][1] + 0.02)

        # Knee: at knee joint
        measure_y(f'knee_{side}',        j[knee_i][1])

        # Calf: 50% between knee and ankle
        measure_y(f'calf_{side}',
                  (j[knee_i][1] + j[ankle_i][1]) / 2)

        # Ankle: 2.5cm above ankle joint
        measure_y(f'ankle_{side}',       j[ankle_i][1] + 0.025)

    # ── Arm circumferences (angled slices perpendicular to arm axis) ──
    for side, sho_i, elb_i, wri_i in [
        ('l', J['l_shoulder'], J['l_elbow'], J['l_wrist']),
        ('r', J['r_shoulder'], J['r_elbow'], J['r_wrist']),
    ]:
        sho, elb, wri = j[sho_i], j[elb_i], j[wri_i]

        # Upper arm: mid shoulder→elbow, slice ⊥ to arm axis
        ua_axis = elb - sho
        ua_axis /= np.linalg.norm(ua_axis)
        lines = _slice(mesh, sho + 0.5 * (elb - sho), ua_axis)
        c = circumference(lines)
        if c: circs[f'upper_arm_{side}'] = c

        # Forearm: mid elbow→wrist, slice ⊥ to forearm axis
        fa_axis = wri - elb
        fa_axis /= np.linalg.norm(fa_axis)
        lines = _slice(mesh, elb + 0.5 * (wri - elb), fa_axis)
        c = circumference(lines)
        if c: circs[f'forearm_{side}'] = c

        # Wrist: 2cm before wrist joint along forearm axis
        lines = _slice(mesh, wri - 0.02 * fa_axis, fa_axis)
        c = circumference(lines)
        if c: circs[f'wrist_{side}'] = c

    # ── Length measurements (joint-to-joint Euclidean) ──
    def dist(a, b): return round(np.linalg.norm(j[a] - j[b]) * 100, 1)
    def chain(*idxs): return round(sum(np.linalg.norm(j[idxs[i+1]] - j[idxs[i]]) for i in range(len(idxs)-1)) * 100, 1)

    # Arm lengths (shoulder → elbow → wrist)
    for side, sho_i, elb_i, wri_i in [
        ('l', J['l_shoulder'], J['l_elbow'], J['l_wrist']),
        ('r', J['r_shoulder'], J['r_elbow'], J['r_wrist']),
    ]:
        lens[f'arm_length_{side}']      = chain(sho_i, elb_i, wri_i)
        lens[f'upper_arm_length_{side}'] = dist(sho_i, elb_i)
        lens[f'forearm_length_{side}']  = dist(elb_i, wri_i)

    # Leg lengths (hip → knee → ankle)
    for side, hip_i, knee_i, ankle_i, foot_i in [
        ('l', J['l_hip'], J['l_knee'], J['l_ankle'], J['l_foot']),
        ('r', J['r_hip'], J['r_knee'], J['r_ankle'], J['r_foot']),
    ]:
        lens[f'inseam_{side}']      = chain(hip_i, knee_i, ankle_i)
        lens[f'outseam_{side}']     = chain(hip_i, knee_i, ankle_i, foot_i)
        lens[f'thigh_length_{side}'] = dist(hip_i, knee_i)
        lens[f'shin_length_{side}']  = dist(knee_i, ankle_i)
        lens[f'leg_length_{side}']   = chain(hip_i, knee_i, ankle_i)

    # Torso / spine (pelvis → spine1 → spine2 → spine3 → neck)
    lens['torso_length'] = chain(
        J['pelvis'], J['spine1'], J['spine2'], J['spine3'], J['neck'])

    # Widths
    lens['shoulder_width'] = dist(J['l_shoulder'], J['r_shoulder'])
    lens['hip_width_bone'] = dist(J['l_hip'],      J['r_hip'])     # bony hip width

    # Crotch height (pelvis to floor level)
    floor_y = min(j[J['l_foot']][1], j[J['r_foot']][1])
    lens['crotch_height'] = round((j[J['pelvis']][1] - floor_y) * 100, 1)

    return {
        'circumferences_cm': {k: v for k, v in circs.items()  if v is not None},
        'widths_cm':         {k: v for k, v in widths.items() if v is not None},
        'depths_cm':         {k: v for k, v in depths.items() if v is not None},
        'lengths_cm':        {k: v for k, v in lens.items()   if v is not None},
    }

print('✅ Measurement functions ready')

In [ ]:
# ── Cell 6: Upload image + enter height ──
from google.colab import files
import io
from PIL import Image

print('Upload a full-body photo (front-facing, person visible head to toe):')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

img_pil  = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')
img_np   = np.array(img_pil)
img_cv2  = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
img_h, img_w = img_cv2.shape[:2]
print(f'Image: {img_w}×{img_h}px')

height_cm = float(input("Person's height in cm (e.g. 175): "))
print(f'Height: {height_cm} cm')

plt.figure(figsize=(5, 8))
plt.imshow(img_np)
plt.axis('off')
plt.title('Input image')
plt.show()

In [ ]:
# ── Cell 7: Run HMR2 inference ──
print('Running HMR2 inference...')

# Use full image as bounding box (single-person assumption)
boxes   = np.array([[0, 0, img_w, img_h]], dtype=float)
dataset = ViTDetDataset(model_cfg, img_cv2, boxes)
loader  = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

batch = next(iter(loader))
batch = recursive_to(batch, device)

with torch.no_grad():
    out = model(batch)

print('✅ Inference done')

# ── Reconstruct neutral-pose SMPL mesh for measurement ──
# We use predicted SHAPE (betas) but NEUTRAL POSE (identity rotations).
# This ensures measurements are consistent regardless of how the person
# is standing in the photo.
betas = out['pred_smpl_params']['betas']   # (1, 10) — person's body shape

with torch.no_grad():
    smpl_out = smpl_neutral(betas=betas, return_verts=True)

vertices = smpl_out.vertices[0].cpu().numpy()   # (6890, 3) in meters
joints   = smpl_out.joints[0].cpu().numpy()     # (45, 3)   in meters
faces    = smpl_neutral.faces                   # (13776, 3)

print(f'Mesh: {len(vertices)} vertices, {len(faces)} faces')
print(f'Shape betas: {betas[0].cpu().numpy().round(3)}')

In [ ]:
# ── Cell 8: Extract all measurements ──
print('Extracting measurements...')
results = extract_all_measurements(vertices, faces, joints, height_cm)

circs  = results['circumferences_cm']
widths = results['widths_cm']
depths = results['depths_cm']
lens   = results['lengths_cm']

print(f'✅ Done — {len(circs)} circumferences, {len(widths)} widths, {len(lens)} lengths')

In [ ]:
# ── Cell 9: Display results ──

def print_section(title, data, unit='cm'):
    print(f'\n  {title}')
    print('  ' + '─' * 40)
    for k, v in sorted(data.items()):
        print(f'    {k:<28} {v:>6.1f} {unit}')

print('=' * 50)
print(f'  BODY MEASUREMENTS  (height: {height_cm} cm)')
print('=' * 50)

print_section('CIRCUMFERENCES', circs)
print_section('WIDTHS (cross-section X-axis)', widths)
print_section('DEPTHS (cross-section Z-axis)', depths)
print_section('LENGTHS', lens)

# Derived health ratios
print('\n  DERIVED RATIOS')
print('  ' + '─' * 40)
waist = circs.get('waist')
hips  = circs.get('hips')
chest = circs.get('chest')
if waist and hips:
    whr = round(waist / hips, 3)
    risk = 'Low' if whr < 0.85 else ('Moderate' if whr < 0.95 else 'High')
    print(f'    Waist-to-Hip Ratio           {whr:>6.3f}  [{risk} risk]')
if chest and waist:
    print(f'    Chest-to-Waist Ratio         {chest/waist:>6.3f}')
sho = lens.get('shoulder_width')
if sho and waist:
    print(f'    Shoulder-to-Waist Ratio      {sho/waist:>6.3f}')
inseam = lens.get('inseam_l') or lens.get('inseam_r')
if inseam:
    print(f'    Inseam-to-Height Ratio       {inseam/height_cm:>6.3f}')
torso = lens.get('torso_length')
if torso:
    print(f'    Cormic Index (torso/height)  {torso/height_cm:>6.3f}')

In [ ]:
# ── Cell 10: Visualize SMPL mesh + measurement slice planes ──
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ── Silhouette view (front + side) ──
fig, axes = plt.subplots(1, 2, figsize=(10, 12))

j = joints[:24] * ((height_cm / 100.0) / (joints[J['head']][1] - min(joints[J['l_foot']][1], joints[J['r_foot']][1])))

for ax, (xi, yi, label) in zip(axes, [(0, 1, 'Front (X-Y)'), (2, 1, 'Side (Z-Y)')]):
    # Draw mesh silhouette (every 10th face for speed)
    v_scaled = vertices * ((height_cm / 100.0) / (
        joints[:24][J['head']][1] - min(joints[:24][J['l_foot']][1], joints[:24][J['r_foot']][1])))
    for face in faces[::20]:
        pts = v_scaled[face]
        ax.fill([pts[0][xi], pts[1][xi], pts[2][xi]],
                [pts[0][yi]*100, pts[1][yi]*100, pts[2][yi]*100],
                color='#c9a882', alpha=0.3, linewidth=0)

    # Draw measurement level lines
    for name, color in [
        ('head', '#e74c3c'), ('neck', '#e67e22'), ('chest', '#f1c40f'),
        ('waist', '#2ecc71'), ('belly', '#1abc9c'), ('hips', '#3498db'),
    ]:
        if name in widths:
            # find y value used for this slice
            pass  # lines would need y lookup — skip for brief viz

    ax.set_aspect('equal')
    ax.set_title(label)
    ax.set_xlabel('cm')
    ax.set_ylabel('cm')
    ax.invert_yaxis()

plt.suptitle(f'SMPL Neutral Mesh  |  height = {height_cm} cm', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11: Export as JSON (matches colab_server.py format) ──
# When integrated into the main pipeline, this block is what the Flask endpoint returns.

import json
from google.colab import files as colab_files

output = {
    'success': True,
    'input': {'height_cm': height_cm},
    'measurements_cm': {
        # ── circumferences (maps to existing colab_server keys) ──
        'chest':      circs.get('chest'),
        'bust':       circs.get('bust'),
        'waist':      circs.get('waist'),
        'belly':      circs.get('belly'),
        'hips':       circs.get('hips'),
        'neck':       circs.get('neck'),
        'head':       circs.get('head'),
        'upper_arm':  circs.get('upper_arm_l'),   # dominant side
        'upper_arm_r': circs.get('upper_arm_r'),
        'forearm':    circs.get('forearm_l'),
        'forearm_r':  circs.get('forearm_r'),
        'wrist':      circs.get('wrist_l'),
        'wrist_r':    circs.get('wrist_r'),
        'thigh':      circs.get('thigh_l'),
        'thigh_r':    circs.get('thigh_r'),
        'thigh_upper':  circs.get('thigh_upper_l'),
        'thigh_upper_r': circs.get('thigh_upper_r'),
        'knee':       circs.get('knee_l'),
        'knee_r':     circs.get('knee_r'),
        'knee_upper': circs.get('knee_upper_l'),
        'calf':       circs.get('calf_l'),
        'calf_r':     circs.get('calf_r'),
        'ankle':      circs.get('ankle_l'),
        'ankle_r':    circs.get('ankle_r'),
    },
    'widths_cm': widths,
    'depths_cm': depths,
    'lengths_cm': lens,
    'betas': betas[0].cpu().tolist(),  # SMPL shape params for future reuse
}
# Remove None values
output['measurements_cm'] = {k: v for k, v in output['measurements_cm'].items() if v is not None}

json_str = json.dumps(output, indent=2)
print(json_str)

# Download the result
with open('measurements_result.json', 'w') as f:
    f.write(json_str)
colab_files.download('measurements_result.json')
print('\n✅ Downloaded measurements_result.json')

---
## Notes

### Why neutral pose?
Cell 7 reconstructs the mesh with **neutral/A-pose** (identity rotations) using only the predicted **shape parameters (β)**. This means measurements don't depend on how the person is standing in the photo — the shape encodes their body proportions, not their pose.

### Measurement accuracy
| Measurement type | Expected accuracy |
|---|---|
| Torso circumferences (chest, waist, hips) | ±2–4 cm |
| Leg circumferences (thigh, calf, knee) | ±2–4 cm |
| Arm circumferences | ±3–5 cm (depends on pose) |
| Lengths (arm, inseam, torso) | ±1–3 cm |
| Widths / depths | ±1–3 cm |

### What's different from the current Colab server
The current `colab_server.py` uses the `Human-Body-Measurements-using-Computer-Vision` repo which gives **8 circumferences** via silhouette regression. This notebook gives **30+ measurements** via 3D mesh slicing. Trade-off: HMR2 requires a GPU and takes ~5s vs ~30s for the full HMR pipeline, but the measurement extraction from mesh adds ~2s.

### Integration path
When you're ready to integrate into `colab_server.py`:
1. Replace the `run_inference()` call with the HMR2 inference block from Cell 7
2. Replace `parse_measurements_from_output()` with `extract_all_measurements()` from Cell 5
3. The `measurements_cm` dict in the response maps directly to what `_enrich_measurements()` in `app.py` already expects